In [1]:
from pathlib import Path

DEX_UP = Path("../use_cases/gse_48328_dexamethasone_up.txt")
GENES = DEX_UP.read_text().split()
print(f"{len(GENES)} genes loaded from {DEX_UP}")

265 genes loaded from ../use_cases/gse_48328_dexamethasone_up.txt


In [2]:
import perturbseqr as ps

DRUG_LIBS = [
    "LINCS L1000 CP",
    "Tahoe-100M",
    "NIBR DRUG-seq",
    "Microarrays CMap",
    "Ginkgo Bioworks",
    "SciPlex",
    "DeepCover MoA",
    "CREEDS Chem",
    "RummaGEO Chem",
]

results = {lib: ps.enrich(GENES, libraries=[lib], top=10) for lib in DRUG_LIBS}
{lib: len(df) for lib, df in results.items()}


{'LINCS L1000 CP': 10,
 'Tahoe-100M': 10,
 'NIBR DRUG-seq': 2,
 'Microarrays CMap': 10,
 'Ginkgo Bioworks': 10,
 'SciPlex': 4,
 'DeepCover MoA': 7,
 'CREEDS Chem': 10,
 'RummaGEO Chem': 10}

## Render the d3 bar graph

`d3.barGraph.js` expects rows in Enrichr's legacy positional format:
`[rank, term, pvalue, zscore, combined_score, "", adj_pvalue]`. 

Place Perturb-Seqr columns onto those positions. The default sort mode reads `d[2]` (pvalue).

In [3]:
def to_legacy_rows(df, n=10):
    df = df.head(n).reset_index(drop=True)
    return [
        [int(i + 1), str(r.term), float(r.pvalue),
         0.0, float(r.oddsRatio or 0), "",
         float(r.adjPvalue if r.adjPvalue is not None else r.pvalue)]
        for i, r in enumerate(df.itertuples())
    ]


In [4]:
import html, json
from pathlib import Path
from IPython.display import display

JS = Path("d3.barGraph.js").read_text()

# ref https://www.kaggle.com/code/dhruvildave/d3-js-jupyter
def render_bar_graph(library):
    df = results[library]
    if df.empty:
        print(f"No enriched terms for {library}")
        return
    rows = to_legacy_rows(df)
    inner = f"""<!doctype html><html><body>
<h3>{html.escape(library)}</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
  jQuery.fn.aToolTip = function () {{ return this; }};
{JS}
  d3.barGraph.createBarGraph({json.dumps(rows)}, "#viz", {{currentMode: function () {{ return ""; }}}});
</script></body></html>"""
    display({"text/html":
        f'<iframe srcdoc="{html.escape(inner, quote=True)}" style="border:none; width:100%; height:380px"></iframe>'
    }, raw=True)


In [5]:
render_bar_graph('LINCS L1000 CP')


<iframe srcdoc="<!doctype html><html><body>
<h3>LINCS L1000 CP</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .at

In [6]:
render_bar_graph('Tahoe-100M')


<iframe srcdoc="<!doctype html><html><body>
<h3>Tahoe-100M</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .attr('

In [7]:
render_bar_graph('NIBR DRUG-seq')


<iframe srcdoc="<!doctype html><html><body>
<h3>NIBR DRUG-seq</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .att

In [8]:
render_bar_graph('Microarrays CMap')


<iframe srcdoc="<!doctype html><html><body>
<h3>Microarrays CMap</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .

In [9]:
render_bar_graph('Ginkgo Bioworks')


<iframe srcdoc="<!doctype html><html><body>
<h3>Ginkgo Bioworks</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .a

In [10]:
render_bar_graph('SciPlex')


<iframe srcdoc="<!doctype html><html><body>
<h3>SciPlex</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .attr('cla

In [11]:
render_bar_graph('DeepCover MoA')


<iframe srcdoc="<!doctype html><html><body>
<h3>DeepCover MoA</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .att

In [12]:
render_bar_graph('CREEDS Chem')


<iframe srcdoc="<!doctype html><html><body>
<h3>CREEDS Chem</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .attr(

In [13]:
render_bar_graph('RummaGEO Chem')


<iframe srcdoc="<!doctype html><html><body>
<h3>RummaGEO Chem</h3>
<div id=viz></div>
<script src=https://code.jquery.com/jquery-3.7.1.min.js></script>
<script src=https://d3js.org/d3.v3.min.js></script>
<script>
 jQuery.fn.aToolTip = function () { return this; };
// from https://github.com/MaayanLab/enrichr/blob/master/src/main/webapp/lib/d3.barGraph.js
/**
 * @desc Visualizes bar graphs and grids. Depends on d3.
 * @params	results: the array of results
 * container: the HTML element that will contain the grid
 * userOptions: configurable options fo rthe bar graph, possible options include:
 * {
 * width: 708,	// width of bar graph in pixels
 * minWidth: 10,	// minimum width of the bar graph in pixels
 * height: 299,	// height of bar graph in pixels
 * bars: 10,	// number of bars to show
 * minColor: '#446688',	// starting minimum color
 * maxColor: '#88ddff',	// starting maximum color
 * modes: [function(d) { return -Math.log(d[2]); }],	// array of functions to apply on results for different modes
 * modeFunction: function(options) {}	// function performed when mode is changed
 * }
 * @author	Edward.Chen@mssm.edu (Edward Y. Chen)
 * @since	10/04/2012
 */
d3.barGraph = {	
	createBarGraph: function(results, container, userOptions) {
 // Don't create bar graph if nowhere to put it
 idTag = container.split(' ')[0];
 if (d3.select(container).empty())
 return;
	
 // Bar graph actually goes in svg-container 
 if (d3.select(container + ' div.svg-container').empty())
 d3.select(container).append('div').classed('svg-container', true);
 container += ' div.svg-container';

 // Bar graph attributes
 var options = {
 width: 708,
 minWidth: 70,
 height: 299,
 bars: 10,
 minColor: '#446688',
 maxColor: '#88ddff',
 modes: [function(d) { return -Math.log(d[2]); }],
 modeFunction: null
 };

 // Copy attributes from userOptions to options
 for (var key in userOptions)
 options[key] = userOptions[key];
 var sortId = container + ' span.method';
 // Initial data
 var filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 var dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Interpolators
 options.x = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .range([options.minWidth, options.width]);
 options.y = d3.scale.ordinal()
 .domain(d3.range(options.bars))
 .rangeBands([0, options.height], .2),
 options.color = d3.scale.linear()
 .domain([d3.min(dataValues), d3.max(dataValues)])
 .interpolate(d3.interpolateRgb)
 .range([options.minColor, options.maxColor]);

 // Create graph
 var chart = d3.select(container)
 .append('svg:svg')
 .datum(options)
 .attr('xmlns', "http://www.w3.org/2000/svg")
 .attr('version', '1.1')
 .attr('width', options.width)
 .attr('height', options.height);

 // Group that contains bar, shadow, and label
 var barGroup = chart.selectAll('g.bar')
 .data(filteredData)
 .enter().append('svg:g')
 .attr('class', 'bar')
 .attr('transform', function(d, i) { return 'translate(0,' + options.y(i) + ')'; });

 d3.barGraph.drawBar(barGroup, options, idTag);

 // Changing mode
 if (options.modes.length < 2) { return; }	// Don't bind click if < 2 modes
 chart.on('click', function() {
 // Change mode
 var mode = options.modes.shift()
 options.modes.push(mode);
 mode = options.modes[0];
 if (options.modeFunction)
 options.modeFunction(options)
 var currentMode = options.currentMode(options);
 
 // Adjust data
 results.sort(function(a, b) { return mode(b) - mode(a); });
 filteredData = results.filter(function(d, i) { 
 return i < options.bars; 
 });
 dataValues = filteredData.map(function(d) { 
 return options.modes[0](d); 
 });

 // Remap interpolators
 options.x.domain([d3.min(dataValues), d3.max(dataValues)]);
 options.color.domain([d3.min(dataValues), d3.max(dataValues)]);

 // Join with existing data and remove old data
 var newBars = chart.selectAll('g.bar').data(filteredData, function(d) { return d; });
 var newBarGroup = newBars.enter().append('svg:g')
 .att